## Creating and Storing Memories in Azure CosmosDB

### Installing Utilities and Libraries

In [ ]:
%pip install azure-cosmos==4.16.0 azure-identity python-dotenv openai==2.38.0

### Setting up the Environment Variables

In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv()

# fetching the cosmosdb configuration from environment variables
cosmosdb_endpoint = os.getenv("COSMOSDB_ENDPOINT")
cosmosdb_key = os.getenv("COSMOSDB_KEY")
database_name = os.getenv("DATABASE_NAME")
container_name = os.getenv("CONTAINER_NAME") + "Vector"

# fetching the azure openai configuration from environment variables
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
embedding_model_name = os.getenv("EMBEDDING_MODEL_NAME")
chat_completions_model_name = os.getenv("CHAT_COMPLETIONS_MODEL_NAME")

### Creating the CosmosDB Client

In [ ]:
from azure.cosmos import CosmosClient
from azure.cosmos import PartitionKey

client = CosmosClient(cosmosdb_endpoint, cosmosdb_key)

### Navigate the Resource Hierarchy

In [ ]:
database = client.get_database_client(database_name)

### Create a Vector Embedding and Indexing Policy in a New Container

In [ ]:
vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path":"/embedding",
            "dataType":"float32",
            "distanceFunction":"cosine",
            "dimensions":1536
        }
    ]
}

vector_indexing_policy = {
    "indexingMode": "consistent",
    "automatic": True,
    "includedPaths": [
        {
            "path": "/*"
        }
    ],
    "excludedPaths": [
        {
            "path": "/_etag/?"
        },
        {
            "path": "/embedding/*"
        }
    ],
    "vectorIndexes": [
        {
        "path": "/embedding",
        "type": "diskANN"
        }
    ],
    "fullTextIndexes": []
}

container = database.create_container_if_not_exists(
    id = container_name,
    partition_key = PartitionKey(path="/customer_id"),
    indexing_policy = vector_indexing_policy,
    vector_embedding_policy = vector_embedding_policy 
)

### Load All 4 Customer Transcripts

In [ ]:
from pathlib import Path

RECORDINGS_FOLDER = Path(
    "Customer-Transcripts"
)


recordings = {}


for file_path in RECORDINGS_FOLDER.glob("*.txt"):

    transcript = file_path.read_text(
        encoding="utf-8"
    )

    recordings[file_path.name] = transcript


print(
    f"Loaded {len(recordings)} recordings"
)

In [ ]:
print(
    recordings[
        "customer_1001_recording.txt"
    ]
)

### Define the Memory Extraction Prompt

In [ ]:
MEMORY_EXTRACTION_PROMPT = """
You are the long-term memory extraction component
of a customer support system.

Analyze the customer support call transcript below.

First, identify the Customer ID mentioned in the transcript.

Then extract information that would be genuinely useful
to a support agent during FUTURE interactions with
this customer.

Good memories include:

- customer communication preferences
- persistent customer preferences
- previous product problems
- recurring issues
- troubleshooting already attempted
- previous resolutions
- important decisions
- unresolved concerns
- relevant customer context

Do NOT store:

- greetings
- generic acknowledgements
- agent pleasantries
- temporary conversational filler
- information with no future support value

Extract at most 5 memories.

Each memory MUST contain:

content:
A concise standalone description of the memory.

memory_type:
One of:
- preference
- previous_issue
- troubleshooting
- resolution
- concern
- customer_context

importance:
Integer between 1 and 10.

Return valid JSON only.

Format:

{{
    "customer_id": "CUST-XXXX",

    "memories": [
        {{
            "content": "...",
            "memory_type": "preference",
            "importance": 7
        }}
    ]
}}

TRANSCRIPT:

{transcript}
"""

### Create the Azure OpenAI Client

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=azure_openai_key,
    api_version="2024-02-15-preview",
    azure_endpoint=azure_openai_endpoint
)

### Create Memories out of the Transcripts

In [ ]:
import json

def extract_memories(
    transcript
):

    prompt = (
        MEMORY_EXTRACTION_PROMPT
        .format(
            transcript=transcript
        )
    )


    response = (
        azure_openai_client.chat.completions.create(
            model=chat_completions_model_name,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0.2,

            response_format={
                "type": "json_object"
            }
        )
    )

    result = json.loads(response.choices[0].message.content)

    return result

### Define the Function to Store Memories

In [ ]:
import uuid
from datetime import datetime, timezone

def store_memory(customer_id, memory, source_file):

    embedding_response = azure_openai_client.embeddings.create(
        model = embedding_model_name,
        input = memory["content"]
    )

    embedding = embedding_response.data[0].embedding


    memory_document = {
        "id": str(uuid.uuid4()),
        "customer_id": customer_id,
        "content": memory["content"],
        "memory_type": memory["memory_type"],
        "importance": memory["importance"],
        "source": source_file,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "embedding": embedding
    }

    container.upsert_item(memory_document)

    print(f"Upserted memory content for Customer ID: {customer_id}")



### Process all 4 Recordings

In [ ]:
for file_name, transcript in recordings.items():
    print("\n==============================")

    print("Processing:", file_name)

    result = extract_memories(transcript = transcript)

    customer_id = result["customer_id"]

    memories = result["memories"]

    for memory in memories:
        store_memory(
            customer_id=customer_id,
            memory=memory,
            source_file=file_name
        )